# Week 5 - fused attention v2, occupancy, fair Triton

Three things changed since Week 4:

1. **Fused attention v2.** v1 gave one Q row to one thread in a 32-thread block, so seq=1024 launched 32 warps for the whole GPU and spilled `acc[64]` to local memory. v2 gives one Q row to one warp in a 128-thread block: 1024 warps, two accumulator floats per lane.
2. **Occupancy reporting without Nsight.** `-Xptxas -v` gives registers and spill bytes at compile time; `cudaFuncGetAttributes` and `cudaOccupancyMaxActiveBlocksPerMultiprocessor` give occupancy at runtime. Neither needs the hardware counters that Colab and RunPod deny.
3. **Triton pinned to IEEE FP32.** The earlier "123% of cuBLAS" on the 4090 was TF32 Tensor Cores on both sides. Both precisions are now reported separately.

**Runtime -> Change runtime type -> T4 GPU**, then run every cell and paste the whole output back.

In [ ]:
import shutil, subprocess, sys
if shutil.which("nvidia-smi") is None:
    sys.exit("No GPU. Runtime -> Change runtime type -> T4 GPU.")
print(subprocess.check_output(["nvidia-smi"], text=True))

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/preethamdandu/cuda-kernel-optimization.git"
DIR="cuda-kernel-optimization"
if [ -d "$DIR/.git" ]; then
  cd "$DIR" && git pull --ff-only
else
  git clone "$REPO_URL" "$DIR"
fi

One script builds and runs everything, so the T4 and the 4090 sessions cannot drift apart. `--quick` skips naive at 4096, which is the only slow case.

In [ ]:
import subprocess

proc = subprocess.run(
    ["bash", "scripts/run_all.sh"],
    cwd="cuda-kernel-optimization",
    text=True,
    capture_output=True,
)
print(proc.stdout)
if proc.stderr:
    print("----- stderr (ptxas register/spill output lives here) -----")
    print(proc.stderr)
print("exit code:", proc.returncode)

Paste both the stdout and the stderr block back into chat. The things to look for:

- `ptxas info` lines: **spill stores/loads** for `fusedAttentionKernel` (v1) versus `fusedAttentionV2Kernel`.
- The occupancy table: **Warps** and **Occupancy** for v1 versus v2.
- `v2/v1` in the attention table: how much the launch geometry alone was worth.
- Whether Triton IEEE FP32 and TF32 now differ by roughly the expected amount.

If the GPU is not a Tesla T4, say so before anything gets written into the README.